# Librerias

In [ ]:
import pandas as pd
import numpy as np
from kmodes.kprototypes import KPrototypes

import matplotlib.pyplot as plt

# Carga de la bbdd y reducción

In [ ]:
#cargar csv
ruta = r""
df =  pd.read_csv("ruta")

# Nos quedamos con las variables que usaremos para segmentar
cols = ["age", "job", "marital", "education", "balance", "deposit", "housing", "loan"]

df_perfil = df[cols].copy()

C:\Users\misab\AppData\Local\Temp\ipykernel_5884\3743423449.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  tablas = pd.read_sql("SHOW TABLES", connection).iloc[:, 0].tolist()
C:\Users\misab\AppData\Local\Temp\ipykernel_5884\3743423449.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dfs[tabla] = pd.read_sql(f"SELECT * FROM {tabla}", connection)


dict_keys(['BANK_marketing'])


# Perfil del cliente

## 1. Preparar datos para K‑Prototipos

In [ ]:
#Convertir variables a 0 y 1 (tal vez ya está)
for col in ["deposit", "housing", "loan"]:
    df_perfil[col] = df_perfil[col].map({"yes": 1, "no": 0})

#Dividir categóricas y numéricas
cat_cols = ["job", "marital", "education"]
num_cols = ["age", "balance", "deposit", "housing", "loan"]

#Matriz que usará K‑Prototipos, reordenado por tipo de variable
data_model = df_perfil[num_cols + cat_cols].copy()

#Guardamos los índices de las columnas categóricas, para saber que se trata como categórica
categorical_indices = [data_model.columns.get_loc(col) for col in cat_cols]

#Convertimos a numpy
X = data_model.to_numpy()

## 2. Entrenar K‑Prototipos

In [ ]:
#Entrenar K‑Prototipos
k = 4  # número de clusters (ajustable)

kproto = KPrototypes(n_clusters=k, init='Cao', random_state=42)
clusters = kproto.fit_predict(X, categorical=categorical_indices)

#Guardamos el cluster en el dataframe original
df_perfil["cluster"] = clusters

## 3. Ver los prototipos de cada cluster

In [ ]:
#Prototipos (centros) de cada cluster
prototypes = kproto.cluster_centroids_

num_prototypes = prototypes[0]  # numéricas
cat_prototypes = prototypes[1]  # categóricas

print("Centros numéricos:\n", num_prototypes)
print("Centros categóricos:\n", cat_prototypes)

## 4. Perfil de productos por cluster

In [ ]:
cluster_products = df_perfil.groupby("cluster")[["deposit","housing","loan"]].mean()
print(cluster_products)

## 5. Perfil demográfico de cada cluster

In [ ]:
#Edad y balance medios:
cluster_num_profile = df_perfil.groupby("cluster")[["age","balance"]].mean()
print(cluster_num_profile)

In [ ]:
#Distribución de job, marital, education:
cluster_job = pd.crosstab(df_perfil["cluster"], df_perfil["job"], normalize="index")
cluster_marital = pd.crosstab(df_perfil["cluster"], df_perfil["marital"], normalize="index")
cluster_education = pd.crosstab(df_perfil["cluster"], df_perfil["education"], normalize="index")

print(cluster_job)
print(cluster_marital)
print(cluster_education)

## 3. Conclusiones